In [0]:
%run ../00-common/config

In [0]:
%run ./00_silver_helpers

In [0]:
from pyspark.sql import functions as F

geo = spark.table(f"{catalog_name}.{bronze_schema}.geolocation")

# nje zip ka disa 'qytete' por jane i njejti qytet me formatim tjeter
geo.filter(F.col("geolocation_zip_code_prefix") == "13455") \
    .groupBy("geolocation_city").count().orderBy(F.desc("count")).show(truncate=False)

In [0]:
from pyspark.sql.window import Window

# normalizo city, hiq ndryshimet e te njejtit qytet, lowercase, trim
geo_norm = geo.withColumn(
    "city_norm",
    F.lower(F.trim(F.translate(
        F.col("geolocation_city"),
        "áàâãäéèêëíìîïóòôõöúùûüç",
        "aaaaaeeeeiiiiooooouuuuc"
    )))
)

# per çdo zip, merr qytetin me te shpeshte
city_rank = (geo_norm.groupBy("geolocation_zip_code_prefix", "city_norm")
    .count()
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("count"))))
    .filter(F.col("rn") == 1)
    .select("geolocation_zip_code_prefix", F.col("city_norm").alias("geolocation_city")))

# mesatare lat/lng + merr shtetin me te shpeshte
geo_coords = (geo.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("geolocation_lat"),
    F.avg("geolocation_lng").alias("geolocation_lng"),
    F.first("geolocation_state").alias("geolocation_state"),
))

# bashko, nje rresht per zip, me qytetin e pastruar
geo_silver = geo_coords.join(city_rank, on="geolocation_zip_code_prefix", how="left")

write_to_silver(geo_silver, "geolocation", catalog_name, silver_schema)